
### 1. What is Unity Catalog?

"Unity Catalog is Databricks' centralized governance solution for managing data and AI assets. It provides centralized access control, data discovery, auditing and lineage across Databricks workspaces. It uses a three-level namespace: catalog, schema and table."

- Simple:
    
    Unity Catalog
         ↓
      Catalog
         ↓
       Schema
         ↓
       Table
    

### 2. Why do we need Unity Catalog?

"We need Unity Catalog to centrally manage data access and governance. Instead of managing permissions separately for different workspaces and tables, we can manage them centrally. It also provides features like access control, auditing, lineage and centralized discovery."

- If they ask "What problem does it solve?":
    "It mainly solves the problem of centralized governance and secure access to data across Databricks environments."

### 3. Explain the 3-level namespace.

The structure is:

catalog.schema.table

For example:

production.sales.customer

Here:
- production → Catalog  
- sales      → Schema  
- customer   → Table  

You can say:
"The catalog is the highest level, schema groups related tables and views, and the table is the actual data object."

### 4. What is a Catalog?

"A catalog is the top-level container in Unity Catalog. We can use catalogs to separate environments, business domains or data zones."

For example:
- dev
- qa
- prod

Or:
- finance
- sales
- marketing

A common enterprise design could be:

prod
 ├── bronze
 ├── silver
 └── gold


### 5. What is a Schema?

"A schema is a logical container inside a catalog that contains objects such as tables, views and other supported objects. We can use schemas to organize data by layer, department or business domain."

Example:

prod.gold.customer

Here:
- prod → catalog
- gold → schema
- customer → table

### 6. What is a Managed Table?

"In a managed table, Databricks manages both the metadata and the underlying data storage location. When the table is dropped, the underlying data can also be removed according to the managed-table behavior and retention policies."

Example:
sql
CREATE TABLE prod.gold.customer (
    customer_id INT,
    name STRING
);

You don't explicitly specify a storage location.

### 7. What is an External Table?

"An external table is a table where the data is stored at a location that we specify, typically in cloud storage such as ADLS. Unity Catalog manages the metadata and access, but the underlying data location is externally specified."

Example:
sql
CREATE TABLE prod.gold.customer
USING DELTA
LOCATION 'abfss://...';


### 8. Managed vs External Table

| Managed                                 | External                                  |
|------------------------------------------|--------------------------------------------|
| Databricks manages table data location   | We specify the storage location            |
| Storage is managed by the platform       | Data remains at the external storage       |
| Easier for Databricks-managed storage    | Useful when data lifecycle is managed      |
| Dropping table can remove underlying data| Dropping the table does not normally delete|

Interview answer:
"The main difference is who manages the underlying data location and lifecycle. With managed tables, Databricks manages the storage location. With external tables, the data stays at a location that we explicitly define."

### 9. What is an External Location?

This is an important one.
"An external location is a Unity Catalog object that combines a cloud storage path with a storage credential. It allows Unity Catalog to control and govern access to a specific external storage location."

Conceptually:

External Location
       │
       ├── ADLS path
       │
       └── Storage Credential

Example concept:

ADLS
abfss://container@storageaccount.dfs.core.windows.net/gold
              ↑
       External Location
              ↑
     Storage Credential


### 10. What is a Storage Credential?

"A storage credential is a Unity Catalog object that contains the authentication information required to access cloud storage. In Azure, this can use an Azure managed identity or service principal, depending on the configuration."

- Important distinction:
    - Storage credential = how Databricks authenticates to storage.
    - External location = where the storage is and which credential is used to access it.

### 11. How do you provide access to ADLS data through Unity Catalog?

A good practical answer:
"First, I configure an appropriate storage credential, typically using an Azure managed identity or service principal. Then I create an external location pointing to the required ADLS path and associate it with that credential. After that, I grant the required Unity Catalog privileges to the appropriate groups or users."

Conceptually:

Azure ADLS
    ↑
Storage Credential
    ↑
External Location
    ↑
Unity Catalog
    ↑
Groups / Users
    ↑
Privileges


### 12. What is RBAC in Unity Catalog?

"RBAC means Role-Based Access Control. Instead of giving permissions individually to every user, we assign users to groups and grant privileges to those groups. This makes access management easier and more secure."

For example:

Finance_Users
     ↓
SELECT
     ↓
Gold tables

And:

Data_Engineers
     ↓
SELECT + MODIFY
     ↓
Bronze / Silver


### 13. How do you grant access to a table?

Example:
sql
GRANT SELECT
ON TABLE prod.gold.customer
TO `finance_users`;


For multiple tables, you can grant permissions at a schema level when appropriate:
sql
GRANT SELECT
ON SCHEMA prod.gold
TO `finance_users`;


Then say:
"I prefer granting permissions to groups rather than individual users because it is easier to manage in an enterprise environment."

### 14. Difference between GRANT SELECT and GRANT MODIFY

- SELECT  
    "SELECT allows the user or group to read data from the table."
- MODIFY  
    "MODIFY allows the user or group to modify the data, such as inserting, updating or deleting data, subject to the required privileges and object permissions."

Simple:

SELECT  → Read
MODIFY  → Change data

Don't say MODIFY means full access. There are other privileges such as CREATE, USE CATALOG, USE SCHEMA, etc.

### 15. How do you restrict access to specific columns?

"For sensitive columns, I can use Unity Catalog's governance capabilities such as column masks and appropriate permissions. A common approach is to create a governed view or apply a column mask so that authorized users see the actual value while other users see a masked value."

Example concept:
| customer_id | name | email        |
|-------------|------|-------------|
| 101         | John | j***@gmail.com |

Instead of exposing:

john@gmail.com


Interview tip  
Don't simply say:  
"I give SELECT only on certain columns."

For fine-grained column-level protection, talk about column masks / governed views depending on the requirement.

### 16. How do you implement row-level security?

"For row-level security, I would use Unity Catalog's supported fine-grained access controls, such as row filters, or a governed view depending on the use case. The filtering logic determines which rows a particular user or group can see."

Example:
- Finance user → Finance records
- Sales user   → Sales records

Conceptually:

              Customer Table
                    ↓
              Row Filter
               ↙       ↘
         Finance      Sales
          users       users


### 17. What is Data Lineage?

"Data lineage shows where data came from, how it was transformed and where it is being consumed. It helps us understand the upstream and downstream dependencies of a data asset."

Example:

ADLS
 ↓
Bronze
 ↓
Silver
 ↓
Gold
 ↓
BI Dashboard

You can trace how the Gold table was created from upstream data.

### 18. How can you track lineage in Unity Catalog?

"Unity Catalog automatically captures lineage for supported Databricks workloads. I can view upstream and downstream lineage from the Unity Catalog or Catalog Explorer interface."

If they ask:  
"Why is lineage useful?"  
"It's useful for impact analysis. For example, before changing a Silver column, I can identify which Gold tables or downstream assets depend on it."

### 19. What is a Metastore?

"A metastore is the top-level governance container for Unity Catalog. It stores metadata and governs catalogs, schemas and other securable objects. Workspaces are attached to a metastore so that they can access the governed data assets."

Think:

Metastore
    ↓
Catalog
    ↓
Schema
    ↓
Table


### 20. Can one workspace use multiple metastores?

For interview purposes, be precise:

"A Databricks workspace is associated with a Unity Catalog metastore for its governance context. The same metastore can be attached to multiple workspaces, which allows them to share access to the same governed data. You generally don't design a workspace to simultaneously use multiple Unity Catalog metastores."

This is better than simply saying "No", because the relationship between workspaces and metastores is important.

### 21. How do you migrate Hive Metastore tables to Unity Catalog?

A good practical answer:
"First, I inventory the existing Hive Metastore tables and understand whether they are managed or external. Then I assess their storage locations and dependencies. For supported tables, I migrate or recreate/register them under Unity Catalog, update the references to the three-level namespace, configure the required storage credentials and external locations, and then apply Unity Catalog permissions. Finally, I validate the data, jobs, permissions and downstream dependencies before moving to production."

If they ask:  
"Would you just change schema.table to catalog.schema.table?"  
Say:  
"Not necessarily. I first need to make sure the underlying storage, table type, permissions and dependencies are compatible with Unity Catalog."

---

## 🔥 Scenario

**Interviewer:**  
"A finance team should access only the Gold tables, while developers should access Bronze and Silver. How would you implement this?"

Strong 3.6-year answer:
"I would separate the data using catalogs or schemas based on the organization's governance requirements. For example, I could have Bronze, Silver and Gold schemas within the production catalog. Then I would create groups such as finance_users and data_engineers. I would grant the finance group SELECT access only to the Gold schema, while the engineering group would have the required read/write privileges on Bronze and Silver. I would avoid giving direct permissions to individual users and manage access through groups."

Example:

prod
│
├── bronze
│     └── raw tables
│
├── silver
│     └── cleansed tables
│
└── gold
      └── business tables

Permissions:

finance_users
      ↓
SELECT
      ↓
prod.gold

data_engineers
      ↓
SELECT + MODIFY
      ↓
prod.bronze
prod.silver

And:
"If Gold contains sensitive data, I would additionally use column masking or row-level security where required."

---

## 🔥 Likely follow-up questions

This is where I would focus for your interview because the interviewer can drill down very quickly.

They may ask:

- **Q:** What's the difference between storage credential and external location?
    - "Storage credential defines how we authenticate to cloud storage, while external location defines the storage path and associates it with that credential."

- **Q:** Can I directly access ADLS from a Databricks notebook without Unity Catalog?
    - "Technically there are other authentication approaches, but in a Unity Catalog governed environment, I would prefer accessing governed data through Unity Catalog rather than distributing storage credentials across notebooks."

- **Q:** Who should get storage credential access?
    - "Normally this is tightly controlled and should not be granted broadly to end users. We use Unity Catalog privileges and external locations to govern access."

- **Q:** Why use groups instead of individual permissions?
    - "It makes access management scalable. When an employee joins or leaves a team, we can manage their group membership instead of changing permissions on every table."

- **Q:** What's the biggest benefit of Unity Catalog?
    - A strong short answer:  
      "Centralized governance—one place to manage access, discovery, lineage and auditing across Databricks data assets."

---

## ⭐ What you should be able to draw on a whiteboard

If they say "Explain how Unity Catalog works in your project", draw:


                  Unity Catalog
                       │
                   Metastore
                       │
                 ┌─────┴─────┐
                 │           │
              Catalog     Catalog
                 │
              prod
                 │
       ┌─────────┼─────────┐
       │         │         │
    bronze     silver     gold
       │         │         │
     Tables    Tables    Tables
       │         │         │
       └─────────┼─────────┘
                 │
        Groups / Permissions
                 │
        ┌────────┴────────┐
        │                 │
   Data Engineers     Finance Users
   Bronze/Silver       Gold only





And separately explain:

ADLS
  ↑
Storage Credential
  ↑
External Location
  ↑
Unity Catalog
  ↑
Tables / Schemas
  ↑
Groups + Privileges


If you can explain this diagram naturally and then answer the follow-up questions above, you'll be in a strong position for the Unity Catalog portion of a Databricks interview.